In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/generative/formatted_apidata.json
/kaggle/input/generative/chat_data_with_retriever.jsonl
/kaggle/input/generative/chat_data_without_retriever.jsonl


In [1]:
!pip install -q transformers datasets accelerate peft bitsandbytes trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 23.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.4/336.4 kB 20.9 MB/s eta 0:00:00


Fine-tune Mistral 7B (or TinyLlama) with QLoRA

In [ ]:
from huggingface_hub import login
login(token="your hugging face access token")

In [13]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
import torch

# Step 1: Load dataset 
dataset_path = "/kaggle/input/generative/chat_data_with_retriever.jsonl"
data = load_dataset("json", data_files=dataset_path, split="train")

# Step 2: Load model & tokenizer 
model_name = "mistralai/Mistral-7B-Instruct-v0.2"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    load_in_4bit=True,
    trust_remote_code=True
)

# Step 3: Prepare QLoRA 
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

# Step 4: Format & tokenize dataset 
def format_and_tokenize(example):
    conversation = example["conversations"]
    prompt = ""
    for turn in conversation:
        prompt += f"{turn['role']}: {turn['content']}\n"
    return tokenizer(prompt, truncation=True, padding="max_length", max_length=1024)

tokenized_dataset = data.map(format_and_tokenize, remove_columns=data.column_names)

# Step 5: Training args 
training_args = TrainingArguments(
    output_dir="/kaggle/working/mistral-finetuned",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    report_to="none"
)

# Step 6: Data collator 
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# Step 7: SFTTrainer
trainer = SFTTrainer(
    model=model,
    train_dataset=tokenized_dataset,
    args=training_args,
    data_collator=data_collator,
)


# Step 8: Train
trainer.train()

# Step 9: Save 
model.save_pretrained("/kaggle/working/mistral-finetuned")
tokenizer.save_pretrained("/kaggle/working/mistral-finetuned")

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

/usr/local/lib/python3.10/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.10/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.10/dist-packages/torch/_dynamo/

Step,Training Loss
10,3.849800
20,1.697700
30,1.547600
40,1.485300
50,1.424200
60,1.200400
70,1.122100
80,1.166300
90,1.090400
100,1.172300


/usr/local/lib/python3.10/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.10/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.10/dist-packages/torch/_dynamo/

('/kaggle/working/mistral-finetuned/tokenizer_config.json',
 '/kaggle/working/mistral-finetuned/special_tokens_map.json',
 '/kaggle/working/mistral-finetuned/tokenizer.model',
 '/kaggle/working/mistral-finetuned/added_tokens.json',
 '/kaggle/working/mistral-finetuned/tokenizer.json')

In [14]:
!pip install -q transformers datasets accelerate peft bitsandbytes trl

In [23]:
pip install faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 51.2 MB/s eta 0:00:00:00:0100:01
Note: you may need to restart the kernel to use updated packages.


In [30]:
pip install -U langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 28.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.3/423.3 kB 26.2 MB/s eta 0:00:00
  Attempting uninstall: async-timeout
    Found existing installation: async-timeout 5.0.1
    Uninstalling async-timeout-5.0.1:
      Successfully uninstalled async-timeout-5.0.1
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.25
    Uninstalling langchain-core-0.3.25:
      Successfully uninstalled langchain-core-0.3.25
  Attempting uninstall: langchain-text-splitters
    Found existing installation: langchain-text-splitters 0.3.3
    Uninstalling langchain-text-splitters-0.3.3:
      Successfully uninstalled langchain-text-splitters-0.3.3
  Attempting uninstall: langchain
    Found existing installation: langchain 0.3.12
    Uninstalling langchain-0.3.12:
      Successfully uninstalled langchain-0.3.

In [31]:
import pandas as pd
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.docstore.document import Document

# Step 1: Load the dataset using `lines=True`
df = pd.read_json("/kaggle/input/generative/formatted_apidata.json", lines=True)

# Step 2: Create Document objects (LangChain format)
docs = [
    Document(page_content=row['prompt'], metadata={"response": row['response']})
    for _, row in df.iterrows()
]

# Step 3: Load Embeddings model
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Step 4: Create FAISS index
db = FAISS.from_documents(docs, embedding_model)

# ✅ Optionally save FAISS index to reuse later
db.save_local("faiss_index")

# ✅ Example retrieval
query = "How to pick a random number in Python?"
retrieved_docs = db.similarity_search(query, k=3)

# Print top responses
for i, doc in enumerate(retrieved_docs):
    print(f"\nResult {i+1}:\nPrompt: {doc.page_content}\nResponse: {doc.metadata['response']}")

<ipython-input-31-bc1cb03479aa>:16: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Result 1:
Prompt: Python | Select random value from a list
Use this API documentation for reference:
API Classes: random, random, random, random
Response: random.choice(), random.randrange(), random.randint(), random.random()

Result 2:
Prompt: Generate random number between 0.1 and 1.0. Python
Use this API documentation for reference:
API Classes: random
Response: random.uniform()

Result 3:
Prompt: Randomly select n elements from list in Python
Use this API documentation for reference:
N/A
Response: are:-seed(), getstate(), choice(), sample()


In [34]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain.docstore.document import Document
import pickle
import pandas as pd

# Load formatted dataset
df = pd.read_json("/kaggle/input/generative/formatted_apidata.json", lines=True)

# Step 1: Extract api_documentation (in this case, the prompt) and build vector embeddings
docs = [Document(page_content=row["prompt"]) for _, row in df.iterrows()]

# Step 2: Generate sentence embeddings using SentenceTransformer
embedding_model = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

# Step 3: Build FAISS index
db = FAISS.from_documents(docs, embedding_model)
index = db.index  # This is the FAISS index you need

# Step 4: Save FAISS index
faiss.write_index(index, "faiss_index.bin")

# Step 5: Save document texts (so we can map index → doc later)
with open("api_doc_texts.pkl", "wb") as f:
    pickle.dump(docs, f)

In [35]:
!pip install rank_bm25 --quiet

from rank_bm25 import BM25Okapi
import pandas as pd
import pickle

# Load the formatted API dataset
df = pd.read_json("/kaggle/input/generative/formatted_apidata.json", lines=True)

# Step 1: Prepare tokenized documents
corpus = [row["prompt"] for _, row in df.iterrows()]
tokenized_corpus = [doc.lower().split() for doc in corpus]

# Step 2: Initialize BM25 retriever
bm25 = BM25Okapi(tokenized_corpus)

# Step 3: Save the corpus and BM25 retriever
with open("bm25_corpus.pkl", "wb") as f:
    pickle.dump(corpus, f)

with open("bm25_model.pkl", "wb") as f:
    pickle.dump(bm25, f)

print(" BM25 retriever and corpus saved.")

✅ BM25 retriever and corpus saved.


In [36]:
import pickle

# Load retriever and corpus
with open("bm25_model.pkl", "rb") as f:
    bm25 = pickle.load(f)

with open("bm25_corpus.pkl", "rb") as f:
    corpus = pickle.load(f)

# Query the retriever
query = "generate a big random number in python"
tokenized_query = query.lower().split()
top_n = bm25.get_top_n(tokenized_query, corpus, n=5)

print(" Top Matches:")
for i, match in enumerate(top_n, 1):
    print(f"{i}. {match}")

 Top Matches:
1. Generate random number between 0.1 and 1.0. Python
Use this API documentation for reference:
API Classes: random
2. How to generate a "big" random number in Python?
Use this API documentation for reference:
API Classes: random
3. Python | Generate random number except K in list
Use this API documentation for reference:
N/A
4. Python – Generate random number except K in list
Use this API documentation for reference:
N/A
5. How do I generate a random 4 digit number and store it as a variable in python?
Use this API documentation for reference:
API Classes: random


In [39]:
import pickle
import faiss
import numpy as np
import pandas as pd
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_community.vectorstores import FAISS
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from sklearn.metrics.pairwise import cosine_similarity

# Load test data
df = pd.read_json("/kaggle/input/generative/formatted_apidata.json", lines=True)
prompts = df["prompt"].tolist()
responses = df["response"].tolist()

# 🔹 Load FAISS index and documents
embedding_model = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
db = FAISS.load_local("faiss_index", embeddings=embedding_model, allow_dangerous_deserialization=True)
with open("/kaggle/working/api_doc_texts.pkl", "rb") as f:
    faiss_docs = pickle.load(f)

# 🔹 Load BM25 model and corpus
with open("/kaggle/working/bm25_model.pkl", "rb") as f:
    bm25 = pickle.load(f)

with open("/kaggle/working/bm25_corpus.pkl", "rb") as f:
    bm25_corpus = pickle.load(f)

# Sentence encoder for similarity checks
encoder = SentenceTransformer("all-MiniLM-L6-v2")

# 🔍 Define comparison function
def compare(query, true_response):
    # --- BM25 ---
    tokenized_query = query.lower().split()
    bm25_top_prompt = bm25.get_top_n(tokenized_query, bm25_corpus, n=1)[0]
    bm25_response = df[df["prompt"] == bm25_top_prompt]["response"].values[0]

    # --- FAISS ---
    faiss_result = db.similarity_search(query, k=1)[0]
    faiss_prompt = faiss_result.page_content
    faiss_response = faiss_result.metadata["response"]

    # --- Similarity ---
    true_vec = encoder.encode([true_response])
    bm25_vec = encoder.encode([bm25_response])
    faiss_vec = encoder.encode([faiss_response])

    sim_bm25 = cosine_similarity(true_vec, bm25_vec)[0][0]
    sim_faiss = cosine_similarity(true_vec, faiss_vec)[0][0]

    return {
        "query": query,
        "true_response": true_response,
        "bm25_prompt": bm25_top_prompt,
        "bm25_response": bm25_response,
        "faiss_prompt": faiss_prompt,
        "faiss_response": faiss_response,
        "bm25_similarity": sim_bm25,
        "faiss_similarity": sim_faiss,
        "bm25_exact_match": bm25_response == true_response,
        "faiss_exact_match": faiss_response == true_response,
    }

# 🔁 Run comparison for first N queries
N = 20
results = [compare(prompts[i], responses[i]) for i in range(N)]

# 📊 Show results
results_df = pd.DataFrame(results)

print("\n=== Accuracy ===")
print("BM25 Exact Match:", results_df["bm25_exact_match"].mean())
print("FAISS Exact Match:", results_df["faiss_exact_match"].mean())

print("\n=== Cosine Similarity (Avg) ===")
print("BM25:", results_df["bm25_similarity"].mean())
print("FAISS:", results_df["faiss_similarity"].mean())

# Optionally show sample comparisons
results_df[["query", "bm25_response", "faiss_response", "true_response", "bm25_similarity", "faiss_similarity"]].head()

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


=== Accuracy ===
BM25 Exact Match: 1.0
FAISS Exact Match: 1.0

=== Cosine Similarity (Avg) ===
BM25: 1.0
FAISS: 1.0


,query,bm25_response,faiss_response,true_response,bm25_similarity,faiss_similarity
0,"How to generate a ""big"" random number in Pytho...",random.randrange(),random.randrange(),random.randrange(),1.0,1.0
1,"Storing big numbers over 9,000 digits in Pytho...",sys.getsizeof(),sys.getsizeof(),sys.getsizeof(),1.0,1.0
2,Python: Calculate factorial of a non-integral ...,math.gamma(),math.gamma(),math.gamma(),1.0,1.0
3,How to get first AND last element of tuple at ...,operator.itemgetter(),operator.itemgetter(),operator.itemgetter(),1.0,1.0
4,PYTHON : Simple random generation driving if/e...,numpy.random.randint(),numpy.random.randint(),numpy.random.randint(),1.0,1.0


In [40]:
from transformers import pipeline
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import SentenceTransformerEmbeddings
import pickle

# Load the HuggingFace pipeline for generation
generator = pipeline("text-generation", model="tiiuae/falcon-rw-1b", max_new_tokens=100)

# Load the FAISS index
embedding_model = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
faiss_db = FAISS.load_local("faiss_index", embeddings=embedding_model, allow_dangerous_deserialization=True)

# Load BM25 retriever and corpus
with open("bm25_model.pkl", "rb") as f:
    bm25 = pickle.load(f)

with open("bm25_corpus.pkl", "rb") as f:
    bm25_corpus = pickle.load(f)

config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.62G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

Device set to use cuda:0


In [41]:
def run_inference(query, mode="zero-shot", retriever="faiss", k=3):
    """
    mode: 'zero-shot' or 'retrieval'
    retriever: 'faiss' or 'bm25'
    """
    if mode == "zero-shot":
        prompt = f"Instruction: {query}\nAnswer:"
    
    elif mode == "retrieval":
        if retriever == "faiss":
            retrieved_docs = faiss_db.similarity_search(query, k=k)
            context = "\n".join([doc.page_content for doc in retrieved_docs])
        
        elif retriever == "bm25":
            tokenized_query = query.lower().split()
            top_n = bm25.get_top_n(tokenized_query, bm25_corpus, n=k)
            context = "\n".join(top_n)
        
        else:
            raise ValueError("Retriever must be 'faiss' or 'bm25'")
        
        prompt = f"Instruction: {query}\n\nRelevant API Docs:\n{context}\n\nAnswer:"

    # Generate response
    result = generator(prompt, do_sample=True, temperature=0.7)[0]["generated_text"]
    return result

In [42]:
query = "How to generate a big random number in Python?"

#  Zero-shot
print("\n Zero-shot:")
print(run_inference(query, mode="zero-shot"))

#  With FAISS retrieval
print("\n FAISS Retrieval:")
print(run_inference(query, mode="retrieval", retriever="faiss"))

#  With BM25 retrieval
print("\n BM25 Retrieval:")
print(run_inference(query, mode="retrieval", retriever="bm25"))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



 Zero-shot:


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Instruction: How to generate a big random number in Python?
Answer:
This is a very common question in Python so I will explain it as best as I can. When you want to generate a random number in Python, you use the rand() function.
The rand function returns a random integer between 0 and the number entered. This function is also known as the randint() function. The rand function is used in data analysis, cryptography, and programming languages.
How to generate a big random number in Python?
To generate a big random number in Python

 FAISS Retrieval:


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Instruction: How to generate a big random number in Python?

Relevant API Docs:
How to generate a "big" random number in Python?
Use this API documentation for reference:
API Classes: random
Generate random number between 0.1 and 1.0. Python
Use this API documentation for reference:
API Classes: random
Fastest way to generate random number from uniform distribtution python
Use this API documentation for reference:
API Classes: numpy.random

Answer:
import numpy as np import time import sys # Random number = np.random.uniform(0, 1) # Random number = np.random.uniform(0.1, 1.0) # Random number = np.random.uniform(0.5, 1) # Random number = np.random.uniform(0.5, 0.9)
How to generate random number between 0.5 and 1.0 in Python?
Use this

 BM25 Retrieval:
Instruction: How to generate a big random number in Python?

Relevant API Docs:
How to generate a "big" random number in Python?
Use this API documentation for reference:
API Classes: random
How do I generate a random 4 digit number and st

In [43]:
!pip install nltk rouge-score sentence-transformers --quiet

  Preparing metadata (setup.py) ... done


In [54]:
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer, util
import nltk
nltk.download("punkt")

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [55]:
# Semantic Similarity
semantic_model = SentenceTransformer('all-MiniLM-L6-v2')

def compute_semantic_similarity(ref, hyp):
    from sentence_transformers import SentenceTransformer
    from sklearn.metrics.pairwise import cosine_similarity

    model = SentenceTransformer("all-MiniLM-L6-v2")
    embeddings = model.encode([ref, hyp])
    return float(cosine_similarity([embeddings[0]], [embeddings[1]])[0][0])

In [56]:
def run_inference(query, mode="retrieval", retriever="faiss", k=1):
    if mode == "zero-shot":
        return "No context used – simulate zero-shot mode."

    if retriever == "faiss":
        results = db.similarity_search(query, k=k)
        return results[0].metadata["response"] if results else "No response found"
    
    if retriever == "bm25":
        tokenized_query = query.lower().split()
        top_n = bm25.get_top_n(tokenized_query, corpus, n=1)
        match_prompt = top_n[0] if top_n else ""
        match_row = df[df["prompt"] == match_prompt]
        return match_row.iloc[0]["response"] if not match_row.empty else "No response found"

    return "Invalid mode/retriever."

In [58]:
# Pick one test query
query = "How to generate a big random number in Python?"

# Find ground truth
matches = df[df['prompt'].str.contains("generate a big random number", case=False)]
ground_truth = matches.iloc[0]['response'] if not matches.empty else "N/A"

# Run retrieval-based inference (FAISS)
generated = run_inference(query, mode="retrieval", retriever="faiss")

# Evaluate using metrics
semantic = compute_semantic_similarity(ground_truth, generated)

# Output the results
print("\n Evaluation Scores:")
print(f"Query: {query}")
print(f"Generated:    {generated}")
print(f"Semantic Sim: {semantic:.4f}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


 Evaluation Scores:
Query: How to generate a big random number in Python?
Generated:    random.randrange()
Semantic Sim: 0.1307


In [62]:
pip install astor

Note: you may need to restart the kernel to use updated packages.


In [66]:
import ast
import astor  # Optional: to convert AST back to code
from typing import Tuple

def parse_to_ast(code: str):
    """Parses code into an AST node."""
    try:
        return ast.parse(code)
    except SyntaxError:
        return None

def ast_equal(ast1, ast2) -> bool:
    """Compare two ASTs structurally."""
    return ast.dump(ast1) == ast.dump(ast2)

def compare_api_calls_ast(generated: str, reference: str) -> Tuple[bool, str]:
    """
    Parse generated and reference API calls and compare using AST.
    Returns (match, reason).
    """
    gen_ast = parse_to_ast(generated)
    ref_ast = parse_to_ast(reference)

    if gen_ast is None or ref_ast is None:
        return False, "Invalid syntax in one of the API calls"

    if ast_equal(gen_ast, ref_ast):
        return True, "ASTs match (API call is correct)"
    else:
        return False, "ASTs do not match (Potential hallucination or syntax mismatch)"

def print_ast(code: str):
    tree = parse_to_ast(code)
    if tree:
        print(ast.dump(tree, indent=4))
    else:
        print("Invalid code")


In [67]:
# Example 1: Correct match
generated_call = "random.randrange()"
reference_call = "random.randrange()"

match, reason = compare_api_calls_ast(generated_call, reference_call)
print("Match:", match)
print("Reason:", reason)

# Example 2: Hallucinated API
generated_call = "random.get_number()"
reference_call = "random.randrange()"

match, reason = compare_api_calls_ast(generated_call, reference_call)
print("Match:", match)
print("Reason:", reason)

Match: True
Reason: ASTs match (API call is correct)
Match: False
Reason: ASTs do not match (Potential hallucination or syntax mismatch)
